# 02 — Compute Sentence-Level MT Metrics (EN-DE / EN-ES)

Scores each MT hypothesis in the WMT24 EN-DE and EN-ES dataset against its human reference using six metrics — COMET, BLEURT, BERTScore, BLEU, chrF, and TER — and writes the enriched data to **`German_Spanish_wmt24_dataset.xlsx`**.

**Reads:** `wmt24_core_ende_enes.xlsx` (two sheets: German, Spanish)  
produced by `latin/01_reproduce_wmt24_core.ipynb`.

Expected columns (from `latin/01`):
```
lang_pair | system | domain | doc | seg_id | source | target | refA
rater | category | severity | error_start | error_end | mqm_score
+ esa_score (Spanish sheet only)
```

**Metrics computed:**

| Metric | Model / version | Notes |
|--------|----------------|-------|
| COMET | `Unbabel/wmt22-comet-da` (unbabel-comet 2.2.6) | Requires `source` + `target` + `refA` |
| BLEURT | `BLEURT-20` (bleurt @ cebe7e6) | Reference-only scoring |
| BERTScore | `microsoft/mdeberta-v3-base` (bert-score 0.3.12) | Returns P / R / F1 |
| BLEU | sacrebleu (sentence-level) | Lexical baseline |
| chrF | sacrebleu (sentence-level) | Morphology-robust lexical |
| TER | sacrebleu (sentence-level) | Lower is better |

**Two scoring paths:**

- **MATEO (recommended for first use):** Upload per-language files to [https://mateo.ivdnt.org/Evaluate](https://mateo.ivdnt.org/Evaluate) and download results. Cell 4 generates files in exactly the format MATEO requires.
- **Local scoring:** Cells 6–9 compute all metrics directly on CPU or GPU using the same model checkpoints.

**Output:** `German_Spanish_wmt24_dataset.xlsx` (two sheets: German, Spanish) — all original columns plus the six metric columns.


## Cell 1 — Install Dependencies

Run once to install `pandas`, `openpyxl`, and `sacrebleu`. Neural metric dependencies (COMET, BLEURT, BERTScore) must be installed in a specific order to avoid version conflicts; the commented block below shows the exact sequence.

In [ ]:
import subprocess, sys

def _install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["pandas", "openpyxl", "sacrebleu"]:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        print(f"Installing {pkg}..."); _install(pkg)

# Neural metric dependencies — install once, then comment out.
# Run in this exact order to avoid version conflicts:
#
# !pip install "transformers==4.40.2"
# !pip install "protobuf==4.25.3"
# !pip install "bert-score==0.3.12" --force-reinstall --no-deps
# !pip install git+https://github.com/google-research/bleurt.git@cebe7e6
# !pip install "unbabel-comet==2.2.6"

print("Dependencies ready.")


## Cell 2 — Imports and Configuration

All paths are relative to the `notebooks/latin/` directory where this notebook lives.

- `INPUT_XLSX` — output of `latin/01_reproduce_wmt24_core.ipynb`.
- `OUTPUT_XLSX` — final enriched file consumed by downstream Latin notebooks.
- `MATEO_DIR` — plain-text exports for [MATEO](https://mateo.ivdnt.org/Evaluate) web scoring (generated in Cell 4).
- `DEVICE` — set to `"cuda"` if a GPU is available; COMET and BLEURT benefit significantly at 10,000+ sentences.
- `SCORE_*` flags allow individual metrics to be skipped without modifying the notebook.

In [ ]:
import os
from pathlib import Path
import warnings
import pandas as pd
warnings.filterwarnings("ignore")

# protobuf fix: must be set before importing TensorFlow / BLEURT
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

# ── Paths ─────────────────────────────────────────────────────────────────────
INPUT_XLSX  = Path("wmt24_core_ende_enes.xlsx")
OUTPUT_XLSX = Path("German_Spanish_wmt24_dataset.xlsx")
MATEO_DIR   = Path("../../data/mateo/latin")
MATEO_DIR.mkdir(parents=True, exist_ok=True)

# ── Column names — must match latin/01 output exactly ─────────────────────────
COL_SRC = "source"
COL_HYP = "target"
COL_REF = "refA"

# ── Language pair config ───────────────────────────────────────────────────────
LANG_PAIRS = {
    "en-de": "German",
    "en-es": "Spanish",
}

# ── Scoring flags ──────────────────────────────────────────────────────────────
SCORE_COMET      = True
SCORE_BLEURT     = True
SCORE_BERTSCORE  = True
SCORE_LEXICAL    = True  # BLEU + chrF + TER (sacrebleu; fast)

# ── Device ─────────────────────────────────────────────────────────────────────
DEVICE = "cpu"  # Options: "cpu" | "cuda"

# ── Model identifiers ──────────────────────────────────────────────────────────
COMET_MODEL     = "Unbabel/wmt22-comet-da"
BLEURT_CKPT     = "BLEURT-20"
BERTSCORE_MODEL = "microsoft/mdeberta-v3-base"

print(f"Input  : {INPUT_XLSX.resolve()}")
print(f"Output : {OUTPUT_XLSX.resolve()}")
print(f"MATEO  : {MATEO_DIR.resolve()}")
print(f"Device : {DEVICE}")


## Cell 3 — Load WMT24 Core Sheets

Reads both sheets from `wmt24_core_ende_enes.xlsx` (written by `latin/01`). Confirms that `source`, `target`, and `refA` are present and non-empty before scoring begins. Rows where `refA` is missing are dropped and counted — these are canary / extra-document rows expected in the WMT24 corpus structure.

In [ ]:
if not INPUT_XLSX.exists():
    raise FileNotFoundError(
        f"{INPUT_XLSX} not found.\n"
        "Run latin/01_reproduce_wmt24_core.ipynb first."
    )

sheets: dict[str, pd.DataFrame] = {}

for lp, sheet_name in LANG_PAIRS.items():
    df = pd.read_excel(INPUT_XLSX, sheet_name=sheet_name)
    for col in [COL_SRC, COL_HYP, COL_REF]:
        if col not in df.columns:
            raise ValueError(
                f"[{sheet_name}] Required column '{col}' missing from {INPUT_XLSX}."
            )
    before = len(df)
    df = df.dropna(subset=[COL_REF]).reset_index(drop=True)
    dropped = before - len(df)
    sheets[lp] = df
    print(f"  {sheet_name:<8} {len(df):,} rows loaded"
          + (f"  ({dropped} dropped — no refA)" if dropped else ""))

print(f"\nSheets ready: {list(sheets.keys())}")


## Cell 4 — Export Files for MATEO (Optional)

[MATEO](https://mateo.ivdnt.org/) computes BERTScore, BLEURT, and COMET without local GPU installation. It expects three plain-text files per language (`source.txt`, `translation.txt`, `reference.txt`), one sentence per line, no header, UTF-8 encoded.

**How to use MATEO:**
1. Go to https://mateo.ivdnt.org/Evaluate
2. Upload `source.txt`, `translation.txt`, and `reference.txt` for one language at a time
3. Select metrics: BERTScore, BLEURT, COMET
4. Click **Evaluate**, download the results CSV
5. Repeat for German and Spanish
6. Place results as `../../data/mateo/latin/<lp>/mateo_results.csv` and run Cell 5

In [ ]:
def export_mateo(df: pd.DataFrame, out_dir: Path,
                 src_col: str, hyp_col: str, ref_col: str) -> None:
    """Write source / translation / reference as plain-text files for MATEO.
    Each file: one sentence per line, no header, UTF-8.
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    for fname, col in [("source", src_col),
                       ("translation", hyp_col),
                       ("reference", ref_col)]:
        series = df[col].fillna("").astype(str)
        series.to_csv(out_dir / f"{fname}.txt",
                      index=False, header=False, encoding="utf-8")


for lp, sheet_name in LANG_PAIRS.items():
    out = MATEO_DIR / lp
    export_mateo(sheets[lp], out, COL_SRC, COL_HYP, COL_REF)
    print(f"  {sheet_name} ({lp}) -> {out}/")

print(f"\nMATEO files ready in: {MATEO_DIR.resolve()}")
print("Upload each subfolder separately to https://mateo.ivdnt.org/Evaluate")


## Cell 5 — Merge Pre-computed MATEO Scores (Optional)

If you scored via MATEO, place the downloaded results CSVs at:
```
../../data/mateo/latin/en-de/mateo_results.csv
../../data/mateo/latin/en-es/mateo_results.csv
```
Run this cell to merge those scores into the working DataFrames. Skip this cell if you are computing metrics locally (Cells 6–9).

Expected MATEO output columns: `COMET`, `BLEURT`, `BERTScore` — adjust `COL_MAP` below if your MATEO version uses different names.

In [ ]:
COL_MAP = {
    "COMET":     "comet",
    "BLEURT":    "bleurt",
    "BERTScore": "bertscore_f1",
}

for lp, sheet_name in LANG_PAIRS.items():
    results_path = MATEO_DIR / lp / "mateo_results.csv"
    if not results_path.exists():
        print(f"  [{sheet_name}] mateo_results.csv not found -- skipped")
        continue
    scores = pd.read_csv(results_path)
    df = sheets[lp].copy()
    for mateo_col, canon_col in COL_MAP.items():
        if mateo_col in scores.columns:
            df[canon_col] = scores[mateo_col].values
    sheets[lp] = df
    print(f"  [{sheet_name}] merged: {list(COL_MAP.values())}")

print("\nMATEO merge complete (skipped language pairs had no results file).")


## Cell 6 — Local Scoring: COMET

Computes COMET scores locally using `Unbabel/wmt22-comet-da` (standard DA model, unbabel-comet 2.2.6). COMET requires `source` + `target` + `refA` for each sentence. Scores are stored on the 0–100 scale used throughout this pipeline.

If `"comet"` is already present in the DataFrame (e.g. from a prior run or Cell 5 MATEO merge), this step is skipped automatically.

- **CPU runtime:** ~15–30 min for EN-DE (~6,000 rows) + EN-ES (~4,600 rows).
- **GPU (CUDA):** ~2–4 min. Set `DEVICE = "cuda"` in Cell 2.

In [ ]:
if SCORE_COMET:
    from comet import download_model, load_from_checkpoint

    gpus = 0 if DEVICE == "cpu" else 1

    print(f"Downloading / loading COMET model: {COMET_MODEL}")
    comet_path  = download_model(COMET_MODEL)
    comet_model = load_from_checkpoint(comet_path)
    print("COMET model ready.\n")

    def score_comet(df: pd.DataFrame) -> list:
        """Return a list of COMET segment scores on the 0–100 scale."""
        records = [
            {"src": str(s), "mt": str(h), "ref": str(r)}
            for s, h, r in zip(df[COL_SRC], df[COL_HYP], df[COL_REF])
        ]
        output = comet_model.predict(records, batch_size=64, gpus=gpus)
        return [round(x * 100, 4) for x in output.scores]

    for lp, sheet_name in LANG_PAIRS.items():
        df = sheets[lp]
        print(f"{'='*55}")
        print(f"{sheet_name} ({lp}) — COMET")
        print(f"{'='*55}")
        if "comet" not in df.columns:
            df["comet"] = score_comet(df)
            print(f"  mean COMET = {df['comet'].mean():.2f}")
        else:
            print("  'comet' already present -- skipped")
        sheets[lp] = df
        print()
else:
    print("SCORE_COMET is False -- skipped.")


## Cell 7 — Local Scoring: BLEURT

Computes BLEURT scores using the `BLEURT-20` checkpoint (~1.2 GB). The model is downloaded automatically on first run.  
Manual download: https://storage.googleapis.com/bleurt-oss-21/BLEURT-20.zip

BLEURT outperforms COMET in human-alignment for EN-DE (Spearman ρ = 0.381 vs 0.323 with MQM) and is the preferred neural metric for EN-DE evaluation in this pipeline.

If `"bleurt"` is already present in the DataFrame, this step is skipped automatically.

In [ ]:
if SCORE_BLEURT:
    from bleurt import score as bleurt_score

    print(f"Loading BLEURT checkpoint: {BLEURT_CKPT}")
    bleurt_scorer = bleurt_score.BleurtScorer(BLEURT_CKPT)
    print("BLEURT scorer ready.\n")

    def score_bleurt(df: pd.DataFrame) -> list:
        """Return a list of BLEURT scores."""
        hyps = df[COL_HYP].fillna("").astype(str).tolist()
        refs = df[COL_REF].fillna("").astype(str).tolist()
        scores = bleurt_scorer.score(references=refs, candidates=hyps, batch_size=64)
        return [round(s, 4) for s in scores]

    for lp, sheet_name in LANG_PAIRS.items():
        df = sheets[lp]
        print(f"{'='*55}")
        print(f"{sheet_name} ({lp}) — BLEURT")
        print(f"{'='*55}")
        if "bleurt" not in df.columns:
            df["bleurt"] = score_bleurt(df)
            print(f"  mean BLEURT = {df['bleurt'].mean():.4f}")
        else:
            print("  'bleurt' already present -- skipped")
        sheets[lp] = df
        print()
else:
    print("SCORE_BLEURT is False -- skipped.")


## Cell 8 — Local Scoring: BERTScore

Computes BERTScore (P / R / F1) using `microsoft/mdeberta-v3-base` (~900 MB). `model_type` is specified explicitly to bypass an AutoModel resolution issue in transformers ≥ 4.41.

BERTScore is the weakest human-alignment metric in this dataset (Spearman ρ ≈ 0.11–0.21 with MQM) and should not be used as a standalone quality proxy. It is included for completeness and intercorrelation analysis.

If `"bertscore_f1"` is already present in the DataFrame, this step is skipped automatically.

In [ ]:
if SCORE_BERTSCORE:
    from bert_score import score as bertscore_fn

    print(f"BERTScore model : {BERTSCORE_MODEL}")
    print(f"Device          : {DEVICE}\n")

    def score_bertscore(df: pd.DataFrame) -> tuple:
        """Return (P, R, F1) lists of BERTScore values."""
        hyps = df[COL_HYP].fillna("").astype(str).tolist()
        refs = df[COL_REF].fillna("").astype(str).tolist()
        P, R, F1 = bertscore_fn(
            cands=hyps,
            refs=refs,
            model_type=BERTSCORE_MODEL,
            verbose=True,
            batch_size=64,
            device=DEVICE,
        )
        return (
            [round(v, 4) for v in P.tolist()],
            [round(v, 4) for v in R.tolist()],
            [round(v, 4) for v in F1.tolist()],
        )

    for lp, sheet_name in LANG_PAIRS.items():
        df = sheets[lp]
        print(f"{'='*55}")
        print(f"{sheet_name} ({lp}) — BERTScore")
        print(f"{'='*55}")
        if "bertscore_f1" not in df.columns:
            P, R, F1 = score_bertscore(df)
            df["bertscore_p"]  = P
            df["bertscore_r"]  = R
            df["bertscore_f1"] = F1
            print(f"  mean F1 = {df['bertscore_f1'].mean():.4f}")
        else:
            print("  'bertscore_f1' already present -- skipped")
        sheets[lp] = df
        print()
else:
    print("SCORE_BERTSCORE is False -- skipped.")


## Cell 9 — Local Scoring: BLEU, chrF, TER (Lexical Metrics)

Computes sentence-level BLEU, chrF, and TER using sacrebleu. These three lexical baselines run entirely on CPU and complete in under 30 seconds.

- **BLEU** — n-gram precision (1–4 grams) with brevity penalty; weakest neural-alignment metric.
- **chrF** — character n-gram F-score; more robust than BLEU for morphologically rich languages such as German.
- **TER** — Translation Edit Rate; lower is better (opposite direction to all other metrics; typical range 30–80).

If `"bleu"` is already present in the DataFrame, all three columns are skipped.

In [ ]:
if SCORE_LEXICAL:
    import sacrebleu as sb

    def score_lexical(df: pd.DataFrame) -> tuple:
        """Return (bleu_list, chrf_list, ter_list) sentence-level scores."""
        hyps = df[COL_HYP].fillna("").astype(str).tolist()
        refs = df[COL_REF].fillna("").astype(str).tolist()
        bleu_scores, chrf_scores, ter_scores = [], [], []
        for h, r in zip(hyps, refs):
            bleu_scores.append(round(sb.sentence_bleu(h, [r]).score, 4))
            chrf_scores.append(round(sb.sentence_chrf(h, [r]).score, 4))
            ter_scores.append(round(sb.TER().corpus_score([h], [[r]]).score, 4))
        return bleu_scores, chrf_scores, ter_scores

    for lp, sheet_name in LANG_PAIRS.items():
        df = sheets[lp]
        print(f"{'='*55}")
        print(f"{sheet_name} ({lp}) — BLEU / chrF / TER")
        print(f"{'='*55}")
        if "bleu" not in df.columns:
            bleu, chrf, ter = score_lexical(df)
            df["bleu"] = bleu
            df["chrf"] = chrf
            df["ter"]  = ter
            print(f"  mean BLEU = {df['bleu'].mean():.2f}")
            print(f"  mean chrF = {df['chrf'].mean():.2f}")
            print(f"  mean TER  = {df['ter'].mean():.2f}  (lower is better)")
        else:
            print("  lexical columns already present -- skipped")
        sheets[lp] = df
        print()
else:
    print("SCORE_LEXICAL is False -- skipped.")


## Cell 10 — Final Column Ordering

Establishes the canonical column order for both sheets. All original `latin/01` columns are preserved; metric columns are appended in the order: neural (COMET, BLEURT, BERTScore P/R/F1) then lexical (BLEU, chrF, TER). Spanish retains `esa_score` from `latin/01` in its original position.

In [ ]:
METRIC_COLS = [
    "comet",
    "bleurt",
    "bertscore_p", "bertscore_r", "bertscore_f1",
    "bleu", "chrf", "ter",
]

for lp, sheet_name in LANG_PAIRS.items():
    df = sheets[lp]
    base_cols = [c for c in df.columns if c not in METRIC_COLS]
    ordered   = base_cols + [c for c in METRIC_COLS if c in df.columns]
    sheets[lp] = df[ordered]
    present = [c for c in METRIC_COLS if c in df.columns]
    print(f"  {sheet_name:<8} columns: {list(sheets[lp].columns)}")
    print(f"           metric cols present: {present}")
    print()


## Cell 11 — Save to Excel

Writes both language pair DataFrames to `German_Spanish_wmt24_dataset.xlsx` as separate sheets (`German`, `Spanish`). This is the file consumed by all downstream Latin notebooks.

In [ ]:
with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    for lp, sheet_name in LANG_PAIRS.items():
        sheets[lp].to_excel(writer, sheet_name=sheet_name, index=False)
        print(f"  Wrote sheet '{sheet_name}' — {len(sheets[lp]):,} rows, "
              f"{len(sheets[lp].columns)} columns")

print(f"\nSaved: {OUTPUT_XLSX.resolve()}")


## Cell 12 — Scoring Summary

Prints mean COMET, BLEURT, BERTScore-F1, BLEU, chrF, and TER for each language pair. Columns not yet computed (e.g. if only some metrics were merged from MATEO) are shown as `--`.

In [ ]:
header = f"{'Lang':<10}  {'COMET':>8}  {'BLEURT':>8}  {'BS-F1':>8}  {'BLEU':>7}  {'chrF':>7}  {'TER':>7}"
print(header)
print("-" * len(header))

for lp, sheet_name in LANG_PAIRS.items():
    df = sheets[lp]

    def _m(col, fmt):
        return format(df[col].mean(), fmt) if col in df.columns else "--"

    print(
        f"  {sheet_name:<8}  "
        f"{_m('comet', '.2f'):>8}  "
        f"{_m('bleurt', '.4f'):>8}  "
        f"{_m('bertscore_f1', '.4f'):>8}  "
        f"{_m('bleu', '.2f'):>7}  "
        f"{_m('chrf', '.2f'):>7}  "
        f"{_m('ter', '.2f'):>7}"
    )

print(f"\n  German_Spanish_wmt24_dataset.xlsx is ready for downstream Latin notebooks.")


## References

- Rei, R., Stewart, C., Farinha, A. C., & Lavie, A. (2020). COMET: A Neural Framework for MT Evaluation. *EMNLP 2020*, pp. 2685–2702. https://aclanthology.org/2020.emnlp-main.213
- Sellam, T., Das, D., & Parikh, A. (2020). BLEURT: Learning Robust Metrics for Text Generation. *ACL 2020*, pp. 7881–7892. https://aclanthology.org/2020.acl-main.704
- Zhang, T., Kishore, V., Wu, F., Weinberger, K. Q., & Artzi, Y. (2020). BERTScore: Evaluating Text Generation with BERT. *ICLR 2020*. https://arxiv.org/abs/1904.09675
- Post, M. (2018). A Call for Clarity in Reporting BLEU Scores. *WMT 2018*. https://aclanthology.org/W18-6319
- Popović, M. (2015). chrF: character n-gram F-score for automatic MT evaluation. *WMT 2015*. https://aclanthology.org/W15-3049
- Snover, M. et al. (2006). A Study of Translation Edit Rate with Targeted Human Annotation. *AMTA 2006*.
- Kocmi, T. et al. (2024). Findings of the WMT24 General MT Shared Task. *WMT 2024*, pp. 1–46. https://aclanthology.org/2024.wmt-1.1/
- Vanroy, B., Tezcan, A., & Macken, L. (2023). MATEO: MAchine Translation Evaluation Online. *EAMT 2023*, pp. 499–500. https://aclanthology.org/2023.eamt-1.52
